# CHIRPS: quickstart

Download one year of CHIRPS daily precipitation (0.25 degree), subset to
East Africa (Kenya and Ethiopia), and plot the annual total.

**Important:** CHIRPS covers 50 S to 50 N only. The UK is outside the
coverage band, so this entry defaults to East Africa (a canonical CHIRPS
use case).

See [`docs/chirps/README.md`](../docs/chirps/README.md) for the full
reference. No authentication. `pip install requests xarray netcdf4
matplotlib cartopy`.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
YEAR = 2023
GRID = "p25"  # p05 for 0.05 degree, p25 for 0.25 degree

# Default region: East Africa (Kenya / Ethiopia)
LAT_SOUTH, LAT_NORTH = -5, 15
LON_WEST, LON_EAST = 34, 42

OUTPUT_DIR = "../data/chirps"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests

print(f"Python       {sys.version.split()[0]}")
for pkg in ["requests", "xarray", "matplotlib", "cartopy", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download one year of CHIRPS daily

Single HTTP GET. At 0.25 degree global daily, the file is ~100 MB per
year. At 0.05 degree it is several GB.

In [ ]:
from scripts.chirps_download import download  # noqa: E402

output_path = download(year=YEAR, grid=GRID, output_dir=OUTPUT_DIR)
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and subset to East Africa

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

# CHIRPS may be served with latitude either ascending or descending.
# sortby guarantees our .sel slice works regardless of which way the axis runs.
ds = ds.sortby("latitude").sortby("longitude")

region = ds["precip"].sel(
    latitude=slice(LAT_SOUTH, LAT_NORTH),
    longitude=slice(LON_WEST, LON_EAST),
)
print(f"\nRegional subset shape: {region.shape}")

## Plot annual total rainfall

Sum daily precipitation over the year to get annual total (mm). For East
Africa in 2023 this highlights the long-rains (March-May) and short-rains
(October-December) totals.

In [ ]:
annual_total = region.sum("time", skipna=True)

fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
annual_total.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="YlGnBu",
    cbar_kwargs={"label": f"Annual total precipitation {YEAR} (mm)"},
)
ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False
ax.set_title(f"CHIRPS annual precipitation, East Africa, {YEAR}")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/chirps/README.md`](../docs/chirps/README.md)
- **Seasonal totals**: resample with `region.resample(time="QS-MAR").sum()`
  for quarterly totals (March-start quarters line up roughly with East
  Africa's long rains, June-August dry season, short rains, etc.).
- **Drought indices**: SPI (Standardised Precipitation Index) is the
  canonical CHIRPS-based index. See the ETO and PyDSTool packages.
- **Higher resolution**: switch `GRID = "p05"` for the 0.05 degree
  product if your region is small and you can tolerate the larger file
  sizes (~3 GB per year).
- **For UK precipitation**: use [ERA5-Land](../notebooks/era5_land_quickstart.ipynb)
  or the Met Office HadUK-Grid product; CHIRPS does not cover the UK.